In [60]:
import pandas as pd
import networkx as nx
from ast import literal_eval
import plotly.graph_objects as go
from tqdm import tqdm

In [36]:
path='/Users/mstudio/Library/CloudStorage/Box-Box/coolie/token/'
df=pd.read_csv(path+'reprint.csv', converters={'pair_state1':literal_eval, 'pair_state2':literal_eval, 'pair_text1':literal_eval, 'pair_text2':literal_eval})
original=pd.read_csv(path+'coolie-token-lemma.csv', converters={'state':literal_eval, 'sent':literal_eval, 'city':literal_eval, 'lemma':literal_eval})
state_df=pd.read_csv('us-state-capitals.csv')

In [3]:
df.shape

(96115, 9)

In [4]:
df['state1']=df['pair_state1'].str[0]
df['state2']=df['pair_state2'].str[0]

In [5]:
df.head(2)

,no_reuse,pair_index1,pair_index2,pair_date1,pair_date2,pair_text1,pair_text2,pair_state1,pair_state2,state1,state2
0,3,1,72742,18910826,18910826,"[tho, steamer, namchow, sail, port, chinese, c...","[bteamer, namchow, sail, port, chinese, coolie...",[New York],[New York],New York,New York
1,3,6,31542,18841025,18841029,"[chinese, general, send, soldier, disguise, co...","[chinese, general, send, soldier, disguise, co...",[Montana],[South Dakota],Montana,South Dakota


In [5]:
interim=pd.merge(df, state_df, left_on='state1', right_on='name')
final=pd.merge(interim, state_df, left_on='state2', right_on='name')

In [6]:
final=final[['state1', 'state2', 'latitude_x', 'longitude_x', 'latitude_y', 'longitude_y']]

In [10]:
final.head(2)

,state1,state2,latitude_x,longitude_x,latitude_y,longitude_y
0,New York,New York,42.652843,-73.757874,42.652843,-73.757874
1,New York,New York,42.652843,-73.757874,42.652843,-73.757874


In [7]:
final.drop(final[final['state1']==final['state2']].index, inplace = True)

In [12]:
final.shape

(72802, 6)

In [8]:
viz_count=final.groupby(['state1', 'state2', 'latitude_x', 'longitude_x', 'latitude_y', 'longitude_y']).size().reset_index(name='count')

In [79]:
viz_count[viz_count['state1']=='Alabama']

,state1,state2,latitude_x,longitude_x,latitude_y,longitude_y,count
0,Alabama,Alaska,32.377716,-86.300568,58.301598,-134.420212,4
1,Alabama,Arizona,32.377716,-86.300568,33.448143,-112.096962,28
2,Alabama,Arkansas,32.377716,-86.300568,34.746613,-92.288986,9
3,Alabama,California,32.377716,-86.300568,38.576668,-121.493629,50
4,Alabama,Colorado,32.377716,-86.300568,39.739227,-104.984856,36
5,Alabama,Connecticut,32.377716,-86.300568,41.764046,-72.682198,24
6,Alabama,Delaware,32.377716,-86.300568,39.157307,-75.519722,5
7,Alabama,Florida,32.377716,-86.300568,30.438118,-84.281296,11
8,Alabama,Georgia,32.377716,-86.300568,33.749027,-84.388229,24
9,Alabama,Hawaii,32.377716,-86.300568,21.307442,-157.857376,14


In [80]:
viz_count[viz_count['state2']=='Alabama']

,state1,state2,latitude_x,longitude_x,latitude_y,longitude_y,count
48,Alaska,Alabama,58.301598,-134.420212,32.377716,-86.300568,2
89,Arizona,Alabama,33.448143,-112.096962,32.377716,-86.300568,16
136,Arkansas,Alabama,34.746613,-92.288986,32.377716,-86.300568,8
184,California,Alabama,38.576668,-121.493629,32.377716,-86.300568,46
232,Colorado,Alabama,39.739227,-104.984856,32.377716,-86.300568,30
279,Connecticut,Alabama,41.764046,-72.682198,32.377716,-86.300568,27
327,Delaware,Alabama,39.157307,-75.519722,32.377716,-86.300568,2
374,Florida,Alabama,30.438118,-84.281296,32.377716,-86.300568,4
420,Georgia,Alabama,33.749027,-84.388229,32.377716,-86.300568,23
468,Hawaii,Alabama,21.307442,-157.857376,32.377716,-86.300568,15


In [ ]:
df_airports = pd.read_csv('https://raw.githubusercontent.com/plotly/datasets/master/2011_february_us_airport_traffic.csv')
df_airports.head()

df_flight_paths = pd.read_csv('https://raw.githubusercontent.com/plotly/datasets/master/2011_february_aa_flight_paths.csv')
df_flight_paths.head()

In [20]:
df_airports.head()

,iata,airport,city,state,country,lat,long,cnt
0,ORD,Chicago O'Hare International,Chicago,IL,USA,41.979595,-87.904464,25129
1,ATL,William B Hartsfield-Atlanta Intl,Atlanta,GA,USA,33.640444,-84.426944,21925
2,DFW,Dallas-Fort Worth International,Dallas-Fort Worth,TX,USA,32.895951,-97.037200,20662
3,PHX,Phoenix Sky Harbor International,Phoenix,AZ,USA,33.434167,-112.008056,17290
4,DEN,Denver Intl,Denver,CO,USA,39.858408,-104.667002,13781


In [21]:
df_flight_paths.head()

,start_lat,start_lon,end_lat,end_lon,airline,airport1,airport2,cnt
0,32.895951,-97.037200,35.040222,-106.609194,AA,DFW,ABQ,444
1,41.979595,-87.904464,30.194533,-97.669872,AA,ORD,AUS,166
2,32.895951,-97.037200,41.938874,-72.683228,AA,DFW,BDL,162
3,18.439417,-66.001833,41.938874,-72.683228,AA,SJU,BDL,56
4,32.895951,-97.037200,33.562943,-86.753550,AA,DFW,BHM,168


In [85]:
import plotly.graph_objects as go
import pandas as pd


fig = go.Figure()

fig.add_trace(go.Scattergeo(
    locationmode = 'USA-states',
    lon = viz_count['longitude_x'],
    lat = viz_count['latitude_x'],
    hoverinfo = 'text',
    text = viz_count['state1'],
    mode = 'markers',
    marker = dict(
        size = 5,
        color = 'rgb(255, 255, 255)',
        line = dict(
            width = 0.5,
            color = 'rgba(255, 255, 255, 255)'
        )
    )))

flight_paths = []
for i in range(len(viz_count)):
    fig.add_trace(
        go.Scattergeo(
            locationmode = 'USA-states',
            lon = [viz_count['longitude_x'][i], viz_count['longitude_y'][i]],
            lat = [viz_count['latitude_x'][i], viz_count['latitude_y'][i]],
            mode = 'lines',
            line = dict(width = 0.0001, color = 'red'),
            # opacity=0.01,
            opacity = float(viz_count['count'][i]) / float(viz_count['count'].max()),
        )
    )

fig.update_layout(
    # title_text = 'Feb. 2011 American Airline flight paths<br>(Hover for airport names)',
    showlegend = False,
    geo = dict(
        center=dict(lon=-100, lat=40),
        scope = 'north america',
        projection_type = 'azimuthal equal area',
        showland = True,
        landcolor = 'rgb(83, 83, 83)',
        countrycolor = 'rgb(83, 83, 83)',
    ),
    autosize=False,  # Disable autosizing
    margin=dict(l=0, r=0, t=0, b=0), 
)

fig.write_image("eacl2023_latex/figure/reprintviz1.pdf", engine='kaleido')
fig.show()

In [14]:
G=nx.from_pandas_edgelist(viz_count, 'state1', 'state2', ['count'])

In [15]:
len(G.nodes())

49

In [16]:
nx.degree_centrality(G)

{'Alabama': 1.0,
 'Alaska': 0.9583333333333333,
 'Arizona': 0.9791666666666666,
 'Arkansas': 1.0,
 'California': 1.0,
 'Colorado': 0.9791666666666666,
 'Connecticut': 1.0,
 'Delaware': 1.0,
 'Florida': 0.9791666666666666,
 'Georgia': 1.0,
 'Hawaii': 1.0,
 'Idaho': 0.9791666666666666,
 'Illinois': 1.0,
 'Indiana': 1.0,
 'Iowa': 1.0,
 'Kansas': 1.0,
 'Kentucky': 0.9791666666666666,
 'Louisiana': 1.0,
 'Maine': 1.0,
 'Maryland': 1.0,
 'Massachusetts': 0.7083333333333333,
 'Michigan': 0.9791666666666666,
 'Minnesota': 1.0,
 'Mississippi': 1.0,
 'Missouri': 1.0,
 'Montana': 1.0,
 'Nebraska': 1.0,
 'Nevada': 1.0,
 'New Jersey': 1.0,
 'New Mexico': 0.9791666666666666,
 'New York': 1.0,
 'North Carolina': 1.0,
 'North Dakota': 1.0,
 'Ohio': 1.0,
 'Oklahoma': 0.9791666666666666,
 'Oregon': 0.9791666666666666,
 'Pennsylvania': 1.0,
 'Rhode Island': 0.9583333333333333,
 'South Carolina': 1.0,
 'South Dakota': 0.9791666666666666,
 'Tennessee': 1.0,
 'Texas': 0.9791666666666666,
 'Utah': 1.0,
 'Ver

In [18]:
from networkx.algorithms import community
nx.average_clustering(G)

0.9905463573033284

In [66]:
df.groupby('pair_index1').count()['no_reuse'].sort_values(ascending=False)[:10]

pair_index1
39       97
8936     94
6439     92
23506    88
29057    83
39796    79
2272     78
43749    77
4197     76
43751    75
Name: no_reuse, dtype: int64

In [68]:
df[df['pair_index1']==39].head(2)

,no_reuse,pair_index1,pair_index2,pair_date1,pair_date2,pair_text1,pair_text2,pair_state1,pair_state2
70,5,39,6331,18760708,18760902,"[discard, german, tolerate, revival, coolie, t...","[discard, germans, tolerate, revival, coolie, ...",[Louisiana],[Missouri]
71,6,39,6439,18760708,18800624,"[discard, german, tolerate, revival, coolie, t...","[german, tolerate, revival, coolie, trade, mon...",[Louisiana],[Ohio]


In [69]:
df[df['pair_index1']==2272].head(2)

,no_reuse,pair_index1,pair_index2,pair_date1,pair_date2,pair_text1,pair_text2,pair_state1,pair_state2
4065,4,2272,3358,19180413,19180114,"[last, week, write, fine, greek, chinese, cool...","[week, write, fine, greek, chinese, coolie, ti...",[Delaware],[Texas]
4066,8,2272,4197,19180413,19180222,"[last, week, write, fine, greek, chinese, cool...","[last, week, write, fine, greek, chinese, cool...",[Delaware],[Montana]


In [71]:
df[df['pair_index1']==4197].head(2)

,no_reuse,pair_index1,pair_index2,pair_date1,pair_date2,pair_text1,pair_text2,pair_state1,pair_state2
6665,6,4197,4557,19180222,19170914,"[last, week, write, fine, greek, chinese, cool...","[write, fine, greek, chinese, coolie, timbucto...",[Montana],[Georgia]
6666,3,4197,4559,19180222,19171108,"[last, week, write, fine, greek, chinese, cool...","[last, week, write, fine, greek, chinese, cool...",[Montana],[Arkansas]


In [39]:
original

,state,city,date,lccn,sent,title,lemma
0,[New York],[New York],18910826,sn83030193,"[., cholera, breaks, out, on, a, steamer, lade...",The evening world. [volume],"[cholera, break, steamer, laden, ohinoso, cool..."
1,[New York],[New York],18910826,sn83030193,"[tho, steamer, namchow, sailed, from, that, po...",The evening world. [volume],"[tho, steamer, namchow, sail, port, chinese, c..."
2,[Montana],[Butte],18830407,sn84036033,"[that, the, authorities, of, kalakaua, 's, kin...",The semi-weekly miner. [volume],"[authority, kalakaua, kingdom, welcome, chines..."
3,[Montana],[Butte],18830407,sn84036033,"[believed, there, is, imminent, danger, of, ly...",The semi-weekly miner. [volume],"[believe, imminent, danger, lynch, kanakas, wa..."
4,[Montana],[Butte],18851114,sn84036033,"[papers, as, the, decedent, may, have, ., stan...",The semi-weekly miner. [volume],"[paper, decedent, may, stand, coolie, london, ..."
...,...,...,...,...,...,...,...
125248,[Maryland],[Thurmont],19200108,sn84026688,"[hie, harm, it, is, claimed, to, work, upon, (...",Catoctin clarion. [volume],"[hie, harm, claim, work, upon, coolie, distanc..."
125249,[New York],[New York],19111029,sn83030272,"[j, tions, ., ,, guid, ., with, porters, and, ...",The sun. [volume],"[j, tion, guid, porter, eighty, coolie, mrs, w..."
125250,[District of Columbia],[Washington],19251015,sn83045462,"[boats, have, been, gecgrksfij, commandeered, ...",Evening star. [volume],"[boats, gecgrksfij, commandeer, forcible, ing,..."
125251,[District of Columbia],[Washington],19250713,sn83045462,"[/p, ), ., —reports, from, changsha, say, a, s...",Evening star. [volume],"[changsha, say, strike, coolie, begin, friday,..."


In [22]:
G.edges['Alabama', 'Alaska']['count']

2